In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from google import genai
from google.genai import types
google_ai_client = genai.Client(api_key=os.getenv("GOOGLE_AI_API_KEY"))

In [6]:
def llm(prompt: str) -> str:
    response = google_ai_client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )
    return response.text

In [7]:
question = 'I just discovered the course. Can i join now?'

In [8]:
context = r'''

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Dont post questions in chat as they may be missed if the room is very active.

edit on GitHub
#Cloud alternatives with GPU
Check the quota and reset cycle carefully. Is the free hours limit per month or per week? Usually, if you change the configuration, the free hours quota might also be adjusted, or it might be billed separately.

Potential options include:

Google Colab
Kaggle
Databricks (possibly)
Consider using GPTs to discover more options. Be aware that some platforms might have restrictions on what you can and cannot install, so ensure to read what is included in the free vs paid tier.

edit on GitHub
#Leaderboard: I am not on the leaderboard / how do I know which one I am on the leaderboard?
When you set up your account, you are automatically assigned a random name, such as “Lucid Elbakyan.” Click on the "Jump to your record on the leaderboard" link to find your entry.

If you want to see what your Display name is, click on the "Edit Course Profile" button.

image #1

First field: This is your nickname/displayed name. You can change it if you want to be known by your Slack username, GitHub username, or any other nickname of your choice. This is useful if you want to remain anonymous.
Second field: Change this to your official name as in your identification documents—passport, national ID card, driver's license, etc. This is mandatory if you do not want "Lucid Elbakyan" on your certificate. This name will appear on your Certificate!
edit on GitHub
#Certificate: Can I follow the course in a self-paced mode and get a certificate?
No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.

edit on GitHub
#I missed the first homework - can I still get a certificate?
Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

edit on GitHub
#Homework: Why does the content keep changing?
This course is being offered for the first time, and things will keep changing until a given module is ready, at which point it shall be announced. Working on the material or homework in advance will be at your own risk, as the final version could be different.

edit on GitHub
#When will the course be offered next?
Summer 2025.

edit on GitHub
#Are there any lectures/videos? Where are they?
Please check the bookmarks and pinned links, especially DataTalks.Clubs YouTube account.

edit on GitHub
#WSL2: ResponseError: model requires more system memory (X.X GiB) than is available (Y.Y GiB). My system has more than X.X GiB.
Your WSL2 is set to use Y.Y GiB, not all your computer memory. To allocate more RAM, follow these steps:

Create a .wslconfig file under your Windows user profile directory: C:\Users\YourUsername\.wslconfig.

Include the desired RAM allocation in the file:

[wsl2]
memory=8GB
Restart WSL using the command:

wsl --shutdown
Run the free command in WSL to verify the changes.

For more details, read this article.

edit on GitHub
#Server Error (500) When logging in to course homework using GitHub
Additional error text seen:

Third-Party Login Failure

An error occurred while attempting to login via your third-party account.
The current solution is to use Google or Slack to log in and submit homework answers, as the root cause analysis for the GitHub issue is sporadic and doesnt impact all users.

edit on GitHub

'''

In [9]:
prompt = f'''

Your task is to answerquestions from the course participants based o the provide context.os
Use the context to find relevant informations and provide accurate answers. If the answer is not found in the context, say "I don't know". 

Question:
{question}

Context:
{context}

'''

In [10]:
answer = llm(prompt)
print(answer)

Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [11]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [12]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 85},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [13]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [14]:
documents[500]

{'id': 'b5d37f88b6',
 'course': 'ai-dev-tools-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How are homework assignments scored?',
 'answer': "Homework grade = points for the questions + 1 point for the FAQ + up to 7 points for learning in public.\n\n- Learning in public: up to 7 points, depending on how many platforms you shared your post on (e.g. LinkedIn, X, blog) and the quality of the post.\n- FAQ: 1 point. To get it, contribute to the FAQ repo and add the link to your PR in your homework submission.\n\nOptional questions are scored too - read the submission form carefully. The homework is for practice and does not affect your certificate. If you think you weren't graded, log in to check your results or search for your name on the leaderboard."}

In [15]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [16]:
search_results = index.search(
    question, 
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"}, 
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [17]:
def search(question, course='llm-zoomcamp'):

    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question, 
        boost_dict=boost_dict,
        filter_dict=filter_dict, 
        num_results=5
    )

In [18]:
search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [19]:
INSTRUCTIONS = """

Your task is to answer questions from the course participants based on the provide context.

Use the context to find relevant informations and provide accurate answers. If the answer is not found in the context, say "I don't know". 

"""

In [20]:
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [21]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [22]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question, 
        context=context)
    return prompt.strip()


In [23]:
prompt = build_prompt(question, search_results)

In [24]:
print(prompt)

Question:
I just discovered the course. Can i join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [25]:
response = google_ai_client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

In [26]:
print(response.model_dump_json(indent=2))

{
  "sdk_http_response": {
    "headers": {
      "x-gemini-service-tier": "standard",
      "content-type": "application/json; charset=UTF-8",
      "vary": "Origin, X-Origin, Referer",
      "content-encoding": "gzip",
      "date": "Mon, 22 Jun 2026 09:54:21 GMT",
      "server": "scaffolding on HTTPServer2",
      "x-xss-protection": "0",
      "x-frame-options": "SAMEORIGIN",
      "x-content-type-options": "nosniff",
      "server-timing": "gfet4t7; dur=2662",
      "alt-svc": "h3=\":443\"; ma=2592000,h3-29=\":443\"; ma=2592000",
      "transfer-encoding": "chunked"
    },
    "body": null
  },
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "media_resolution": null,
            "code_execution_result": null,
            "executable_code": null,
            "file_data": null,
            "function_call": null,
            "function_response": null,
            "inline_data": null,
            "text": "Yes, you can still join! \n\nHowever, if 

In [27]:
response.text

"Yes, you can still join! \n\nHowever, if you want to receive a certificate, you must submit your project while submissions are still being accepted. You don't need to wait for a registration confirmation to start—you can begin learning, watching the course videos, and submitting homework right away (as long as the submission forms are still open)."

In [28]:
response.usage_metadata

GenerateContentResponseUsageMetadata(
  candidates_token_count=71,
  prompt_token_count=514,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=514
    ),
  ],
  thoughts_token_count=384,
  total_token_count=969
)

In [29]:
response = google_ai_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        {"role": "user", "parts": [{"text": prompt}]}
    ],
    config=types.GenerateContentConfig(
        system_instruction=INSTRUCTIONS
    )
)

In [30]:
response.text

'Yes, you can join now. You can start whenever you want, and the videos and GitHub materials are available. If you want to receive a certificate, you will need to submit your project while submissions are still being accepted.'

In [31]:
def llm(instructions, user_prompt, model="gemini-2.5-flash"):

    response = google_ai_client.models.generate_content(
        model=model,
        contents=[
            {"role": "user", "parts": [{"text": user_prompt}]}
        ],
        config=types.GenerateContentConfig(
            system_instruction=instructions
        )
    )

    return response.text

In [32]:
def rag(query, model='gemini-2.5-flash'):
    search_results = search(query)
    user_prompt = build_prompt(query, search_results)
    return llm(INSTRUCTIONS, user_prompt, model=model)

In [33]:
answer = rag(question)
print(answer)

Yes, you can still join the course. You can start whenever you want, as the videos and GitHub materials are available.

However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [34]:
def search(query, course='llm-zoomcamp'):

    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        query, 
        boost_dict=boost_dict,
        filter_dict=filter_dict, 
        num_results=5
    )

In [35]:
search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search",
            description="Search the FAQ database for entries matching the given query.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "query": types.Schema(
                        type=types.Type.STRING,
                        description="Search query text to look up in the course FAQ."
                    )
                },
                required=["query"]
            )
        )
    ]
)

In [36]:
response = google_ai_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        {"role": "user", "parts": [{"text": prompt}]}
    ],
    config=types.GenerateContentConfig(
        system_instruction=INSTRUCTIONS,
        tools=[search_tool],
        tool_config=types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(
                mode="ANY",
            )
        )
    ),
)

In [37]:
response

GenerateContentResponse(
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            function_call=FunctionCall(
              args=<... Max depth ...>,
              name=<... Max depth ...>
            ),
            thought_signature=b'\n\xaa\x03\x01\x0c9\xd6\xc7z\xe2\xe1\xa3\xf2\x06}\x93\x0c\x96\xab\xbd\x06\xd9\xb3\x9e\x0e\xa6\xea\xdc}\xab\xffdJ9e)3\xfe\xcc\xaco\x06`\x9e\xbf\xd3\xb7\xf4\x82\x18WN,\xe3W\xcb\xec!\x7fN\x7fM\x0eI\xdf\xcbY\xb1\x08\xc1\x9b\xd0t#I\xc4a\x9f#\x97\xad\x19\x90\x19\xbf7e"t\xb5k\x0e\x8d\xc3\xb1\xb3\xa6...'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  response_id='UQY5apHEEKuHnsEP68HU-Ao',
  sdk_http_response=HttpResponse(
    headers=<dict len=12>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=19,
    prompt_token_count=616,
    prompt_tokens_details=[
      Modalit

In [38]:
call = response.candidates[0].content.parts[0].function_call
call

FunctionCall(
  args={
    'query': 'Can I still join the course?'
  },
  name='search'
)

In [39]:
call.args

{'query': 'Can I still join the course?'}

In [40]:
call.id 

In [41]:
results = search(**call.args)

In [42]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\n

In [43]:
import json
result_json = json.dumps(results, indent=2)
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "9f689c185f",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I missed the first homework - can I still get a certificate?",
    "answer": "Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certifi

In [44]:
function_response_part = types.Part(
    function_response=types.FunctionResponse(
        id=call.id,       
        name=call.name,
        response={"result": result_json}
    )
)

In [45]:
follow_up = google_ai_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        {"role": "user",  "parts": [{"text": prompt}]},
        {"role": "model", "parts": [types.Part(function_call=call)]},
        {"role": "user",  "parts": [function_response_part]},
    ],
    config=types.GenerateContentConfig(
        system_instruction=INSTRUCTIONS,
        tools=[search_tool]
    ),
)

print(follow_up.text)  # resposta final do modelo

Yes, you can still join the course. However, if you wish to receive a certificate, you must submit your project while submissions are still being accepted.


In [46]:
follow_up.usage_metadata 

GenerateContentResponseUsageMetadata(
  candidates_token_count=31,
  prompt_token_count=1327,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=1327
    ),
  ],
  thoughts_token_count=49,
  total_token_count=1407
)

In [47]:
def make_call(call):
    
    results = search(**call.args)

    result_json = json.dumps(results, indent=2)

    return types.Part(
        function_response=types.FunctionResponse(
            id=call.id,       
            name=call.name,
            response={"result": result_json}
        )
    )

In [48]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "model", "parts": [{"text": instructions}]},
    {"role": "user", "parts": [{"text": question}]}
]

In [49]:
response = google_ai_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=messages,
    config=types.GenerateContentConfig(
        tools=[search_tool]
    ),
)

In [50]:
response

GenerateContentResponse(
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            function_call=FunctionCall(
              args=<... Max depth ...>,
              name=<... Max depth ...>
            ),
            thought_signature=b"\n\x99\x02\x01\x0c9\xd6\xc7V\xa1\xa6\xeaG\x9b\xa5W\xeas\x08\x80-\x98\xb8o\xa9|\x06\x1a\xe59\r\xed{\xad\xff\xd5j\r_\xc9\x1e\x02\xdbj\x7fe\\m0\x89w'\x15\x13/\xa8\x99\x00\x9fu\xd6-\x16\xb3\xb5\xb8\xe9%\xee\x9d>\x80\x86Cc\xe9\xc9\x9b\x05giV\x91\xcd\x16\xb0.=\x8f\xc8\x8c\xdfm\x8a\x84\xe9T...'
          ),
          Part(
            function_call=FunctionCall(
              args=<... Max depth ...>,
              name=<... Max depth ...>
            )
          ),
          Part(
            function_call=FunctionCall(
              args=<... Max depth ...>,
              name=<... Max depth ...>
            )
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      in

In [51]:
messages.extend(response.candidates[0].content.parts)

for item in response.candidates[0].content.parts:
    if item.function_call:
        print('function_call:', item.function_call)
        call_output = make_call(item.function_call)
        messages.append(call_output)
    elif item.text:
        print('ASSISTANT:')
        print(item.text)
        

function_call: id=None args={'query': 'join course'} name='search' partial_args=None will_continue=None
function_call: id=None args={'query': 'enroll course'} name='search' partial_args=None will_continue=None
function_call: id=None args={'query': 'late enrollment'} name='search' partial_args=None will_continue=None


In [52]:
messages

[{'role': 'model',
  'parts': [{'text': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."}]},
 {'role': 'user',
  'parts': [{'text': 'I just discovered the course. Can I join it?'}]},
 Part(
   function_call=FunctionCall(
     args={
       'query': 'join course'
     },
     name='search'
   ),
   thought_signature=b"\n\x99\x02\x01\x0c9\xd6\xc7V\xa1\xa6\xeaG\x9b\xa5W\xeas\x08\x80-\x98\xb8o\xa9|\x06\x1a\xe59\r\xed{\xad\xff\xd5j\r_\xc9\x1e\x02\xdbj\x7fe\\m0\x89w'\x15\x13/\xa8\x99\x00\x9fu\xd6-\x16\xb3\xb5\xb8\xe9%\xee\x9d>\x80\x86Cc\xe9\xc9\x9b\x05giV\x91\xcd\x16\xb0.=\x8f\xc8\

In [53]:
messages = [
    {"role": "model", "parts": [{"text": instructions}]},
    {"role": "user", "parts": [{"text": question}]}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = google_ai_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=messages,
        config=types.GenerateContentConfig(
            tools=[search_tool]
        ),
    )

    messages.extend(response.candidates[0].content.parts)

    for item in response.candidates[0].content.parts:
        if item.function_call:
            print('function_call:', item.function_call)
            call_output = make_call(item.function_call)
            messages.append(call_output)
            has_function_calls = True
        elif item.text:
            print('ASSISTANT:')
            print(item.text)

    if has_function_calls == False:
        break
    it += 1

iteration #1...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 31.397526324s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'},

In [66]:
def agent_loop (instructions, question, model = "gemini-3.1-flash-lite") -> str:
    messages = [
        {"role": "model", "parts": [{"text": instructions}]},
        {"role": "user", "parts": [{"text": question}]}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = google_ai_client.models.generate_content(
            model=model,
            contents=messages,
            config=types.GenerateContentConfig(
                tools=[search_tool]
            ),
        )

        messages.extend(response.candidates[0].content.parts)

        for item in response.candidates[0].content.parts:
            if item.function_call:
                print('function_call:', item.function_call)
                call_output = make_call(item.function_call)
                messages.append(call_output)
                has_function_calls = True
            elif item.text:
                print('ASSISTANT:')
                last_answer = item.text
                print(item.text)

        if has_function_calls == False:
            break
        it += 1

    return last_answer

In [55]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform serche, analyze the results, and then perform more searches.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

In [56]:
agent_loop(instructions, question)

iteration #1...
function_call: id='tnk5u1xi' args={'query': 'Can I join the course'} name='search' partial_args=None will_continue=None
iteration #2...
function_call: id='6n9inzg7' args={'query': 'register link course platform'} name='search' partial_args=None will_continue=None
iteration #3...
ASSISTANT:
Yes, you can absolutely still join! 

Here is what you need to know to get started:

### 1. Can I still get a certificate?
Yes, but you will need to submit your project while we are still actively accepting submissions. 
* *Note:* You can only earn a certificate if you finish the course with the "live" cohort, as you must peer-review 3 capstone projects after submitting yours. Peer reviews can only be done while the live course is actively running. Certificates are not awarded for the self-paced mode.

### 2. Do I need to wait for registration approval?
No. Registration is mainly to gauge initial interest. You are automatically accepted, and you do not need to wait for a confirmation 

'Yes, you can absolutely still join! \n\nHere is what you need to know to get started:\n\n### 1. Can I still get a certificate?\nYes, but you will need to submit your project while we are still actively accepting submissions. \n* *Note:* You can only earn a certificate if you finish the course with the "live" cohort, as you must peer-review 3 capstone projects after submitting yours. Peer reviews can only be done while the live course is actively running. Certificates are not awarded for the self-paced mode.\n\n### 2. Do I need to wait for registration approval?\nNo. Registration is mainly to gauge initial interest. You are automatically accepted, and you do not need to wait for a confirmation email. You can register on the course platform and start learning and submitting homework immediately (while the submission forms are open).\n\n### 3. How to start:\n* **Resources:** Start by reviewing the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomca

In [68]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform serche, analyze the results, and then perform more searches.

The question has to be about the course or its logistics, offtopic questions shouldn't be answered.
If the search returns nothing, it's likely an off-topic question. If you can't answer the question using FAQ, don't do it yourself.
Only use the facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()


question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: id='F9Z57OvZ' args={'query': 'what is queen gambit'} name='search' partial_args=None will_continue=None
iteration #2...
ASSISTANT:
I'm sorry, but I don't have information about "Queen's Gambit" in the course FAQ. It appears to be off-topic.

Are there any other areas regarding the course that you would like to explore?
